# WASB on the Volleyball dataset: ONNX export and evaluation

WASB (Tarashima et al., BMVC 2023; MIT licence) publishes ball-detection weights trained on the Volleyball dataset.
This notebook (1) builds the model with WASB's own code and loads those weights, (2) exports it to ONNX with the
sigmoid included (`/kaggle/working/wasb_volleyball.onnx`, used by `vball.ball_detect.Wasb`), checking ONNX against
PyTorch, (3) evaluates it with the project's script (`scripts/eval_ball_detection.py --detectors wasb`, the same metric
and test clips as the other detectors) and (4) runs WASB's own evaluation for reference.

Runs on **Kaggle** (GPU, Internet on) with input `sherif31/group-activity-recognition-volleyball`.

In [ ]:
import glob, os
WORK = '/tmp/work'; OUT = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)
VIDEOS = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/videos/47', recursive=True))
!git clone -q --depth 1 https://github.com/nttcom/WASB-SBDT {WORK}/wasb
!git clone -q --depth 1 https://github.com/DL-Volleyball-Analysis/volleyball-analysis {WORK}/repo
!pip -q install -e {WORK}/repo --no-deps
!pip -q install onnx onnxruntime-gpu gdown hydra-core omegaconf
!mkdir -p {WORK}/wasb/pretrained_weights && gdown -q 1M9y4wPJqLc0K-z-Bo5DP8Ft5XwJuLqIS -O {WORK}/wasb/pretrained_weights/wasb_volleyball_best.pth.tar
!gdown -q 1urZpZiiepC85JD1u3VeURgUpztRgI0yl -O {WORK}/ball_ann.zip && unzip -q -o {WORK}/ball_ann.zip -d {WORK}/ann
ANN = f'{WORK}/ann/volleyball_ball_annotation'

In [ ]:
import sys, torch, numpy as np
from omegaconf import OmegaConf
sys.path.insert(0, f'{WORK}/wasb/src')
from models import build_model
cfg = {'model': OmegaConf.load(f'{WORK}/wasb/src/configs/model/wasb.yaml')}  # HRNet reads cfg.MODEL (attribute access)
net = build_model(cfg)
ckpt = torch.load(f'{WORK}/wasb/pretrained_weights/wasb_volleyball_best.pth.tar', map_location='cpu', weights_only=False)
net.load_state_dict(ckpt['model_state_dict']); net.eval()

class Export(torch.nn.Module):   # 3 RGB frames (9 channels) -> 3 heatmaps in [0, 1]
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x)[0].sigmoid()

x = torch.randn(1, 9, 288, 512)
ONNX = f'{OUT}/wasb_volleyball.onnx'
torch.onnx.export(Export(net), x, ONNX, input_names=['frames'], output_names=['heatmaps'], opset_version=17,
                  dynamic_axes={'frames': {0: 'batch'}, 'heatmaps': {0: 'batch'}})
import onnxruntime as ort
y_onnx = ort.InferenceSession(ONNX, providers=['CPUExecutionProvider']).run(None, {'frames': x.numpy()})[0]
with torch.no_grad(): y_torch = Export(net)(x).numpy()
print('params', sum(p.numel() for p in net.parameters()), 'max |onnx - torch|', float(np.abs(y_onnx - y_torch).max()))
!mkdir -p {WORK}/repo/models && cp {ONNX} {WORK}/repo/models/wasb_volleyball.onnx

In [ ]:
!cd {WORK}/repo && python scripts/eval_ball_detection.py --videos {VIDEOS} --ann {ANN} --detectors wasb --out {OUT}/ball_eval_wasb
# held-out gym matches (private dataset yoga0702/vball-orel: volleyball-orel.ru data, research use)
OREL = next(iter(glob.glob('/kaggle/input/**/data/test', recursive=True)), None)
if OREL:
    OREL = os.path.dirname(os.path.dirname(OREL))
    !cd {WORK}/repo && python scripts/eval_ball_detection.py --dataset orel --orel {OREL} --detectors wasb v4c --out {OUT}/ball_eval_orel

In [ ]:
# WASB's own evaluation (its configs expect /root/datasets/volleyball/{videos, volleyball_ball_annotation})
try:
    os.makedirs('/root/datasets/volleyball', exist_ok=True)
    for name, src in (('videos', VIDEOS), ('volleyball_ball_annotation', ANN)):
        dst = f'/root/datasets/volleyball/{name}'
        if not os.path.exists(dst): os.symlink(src, dst)
    !cd {WORK}/wasb/src && python main.py --config-name=eval dataset=volleyball model=wasb detector.model_path=../pretrained_weights/wasb_volleyball_best.pth.tar 2>&1 | tee {OUT}/wasb_own_eval.txt | tail -25
except Exception as e:
    print('WASB own evaluation failed:', e)